# Section 2 — Euler's Method

[← Direction Fields](01_direction_fields_autonomous_odes.ipynb) · [Module 3](README.md) · [Next: Euler Error →](03_euler_error.ipynb)

Euler's method converts the tangent-line idea into a numerical algorithm for an IVP

$$
y'=f(t,y),\qquad y(t_0)=y_0.
$$

With step size $h$, define $t_n=t_0+nh$. If $w_n\approx y(t_n)$, Euler's method uses

$$
\boxed{w_{n+1}=w_n+h f(t_n,w_n)}.
$$

### Learning goals

- derive Euler's update from a tangent line;
- implement Euler's method from scratch;
- compare a numerical approximation with an exact solution;
- investigate how the step size changes the answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. A reusable Euler solver

The algorithm only needs the function $f$, an initial value, a step size, and a number of steps.

In [ ]:
def euler(f, t0, y0, h, n_steps):
    t = np.empty(n_steps + 1, dtype=float)
    w = np.empty(n_steps + 1, dtype=float)
    t[0], w[0] = t0, y0

    for n in range(n_steps):
        w[n + 1] = w[n] + h * f(t[n], w[n])
        t[n + 1] = t[n] + h

    return t, w

## 2. Test IVP

Use the IVP

$$
y'=t e^{3t}-2y,\qquad y(0)=0,
$$

on $0\le t\le1$. Its exact solution is

$$
y(t)=e^{3t}\left(\frac{t}{5}-\frac{1}{25}\right)+\frac{1}{25}e^{-2t}.
$$

We begin with the deliberately large step size $h=0.5$, so there are only two Euler steps.

In [ ]:
def f(t, y):
    return t * np.exp(3*t) - 2*y


def exact(t):
    return np.exp(3*t) * (t/5 - 1/25) + np.exp(-2*t)/25

h = 0.5
t, w = euler(f, t0=0.0, y0=0.0, h=h, n_steps=2)

print(" n      t_n        w_n        exact       error")
for n, (tn, wn) in enumerate(zip(t, w)):
    err = abs(exact(tn) - wn)
    print(f"{n:2d}   {tn:6.2f}   {wn:10.6f}   {exact(tn):10.6f}   {err:10.6f}")

Notice the first step: because $f(0,0)=0$, Euler's method initially predicts no change. The true solution curves upward between the mesh points, so a large step size misses important behavior.

In [ ]:
t_fine = np.linspace(0, 1, 400)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(t_fine, exact(t_fine), linewidth=2, label="exact solution")
ax.plot(t, w, marker="o", linewidth=2, label="Euler, h=0.5")
ax.set_xlabel("t")
ax.set_ylabel("y")
ax.set_title("Euler approximation versus the exact solution")
ax.legend()
ax.grid(True, alpha=0.25)
plt.show()

## 3. What happens when we reduce $h$?

For a fixed interval, smaller $h$ means more steps and more work, but the tangent-line approximation is refreshed more frequently.

In [ ]:
step_sizes = [0.5, 0.25, 0.1, 0.05]

print("    h    steps      Euler y(1)       exact y(1)       error")
for h in step_sizes:
    n_steps = round(1 / h)
    t_h, w_h = euler(f, 0.0, 0.0, h, n_steps)
    error = abs(exact(1.0) - w_h[-1])
    print(f"{h:6.3f}   {n_steps:4d}      {w_h[-1]:12.8f}    {exact(1.0):12.8f}   {error:10.6f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(t_fine, exact(t_fine), linewidth=2, label="exact")

for h in [0.5, 0.25, 0.1]:
    t_h, w_h = euler(f, 0.0, 0.0, h, round(1/h))
    ax.plot(t_h, w_h, marker="o", linewidth=1.5, label=f"Euler h={h}")

ax.set_xlabel("t")
ax.set_ylabel("y")
ax.set_title("Smaller step sizes track the solution more closely")
ax.legend()
ax.grid(True, alpha=0.25)
plt.show()

## 4. The geometric interpretation

At each mesh point $(t_n,w_n)$:

1. evaluate the slope $f(t_n,w_n)$;
2. move horizontally by $h$;
3. move vertically by $h f(t_n,w_n)$.

Euler's method is therefore a chain of tangent-line approximations. It is simple and inexpensive, but errors from each step accumulate.

### Try it

Change the IVP or the step size and rerun the notebook. In particular, compare $h=0.5$, $0.25$, $0.125$, and $0.0625$. What pattern do you see in the error at $t=1$?

That question leads directly to the next section.

[← Direction Fields](01_direction_fields_autonomous_odes.ipynb) · [Next: Euler Error →](03_euler_error.ipynb)